# Blocking: generating candidate pairs

Every method so far has quietly assumed that comparing all pairs is possible. At
the size of these two registers it is: 30,000 times 27,000 is 810 million
comparisons, which a laptop will grind through. At national scale it is not
possible at all, by a wide margin.

**Blocking** is how the problem is made finite: only compare records that already
agree on something. It is also the single most consequential design decision in a
linkage pipeline, because a match excluded by blocking is a match no model will
ever see, no matter how good the model is.

**What you will do**

1. Quantify the scale problem
2. Measure a blocking rule with three standard metrics
3. Build simple, conjunctive, disjunctive and phonetic rules and compare them
4. Find the recall ceiling that a rule set imposes
5. Try the sorted neighbourhood method, which works differently

## 0. Setup and prepared data

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 130)

DATA = Path("../../data")
if not DATA.exists():
    DATA = Path("data")

fonasa = pd.read_csv(DATA / "fonasa_sample.csv", dtype=str)
suseso = pd.read_csv(DATA / "suseso_sample.csv", dtype=str)


def basic_text_clean(series):
    return (series.astype("string").str.strip().str.upper()
            .str.replace(r"\s+", " ", regex=True))


def remove_accents(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(c for c in value if not unicodedata.combining(c))


def standardise_name(series):
    cleaned = basic_text_clean(series)
    cleaned = cleaned.map(remove_accents, na_action="ignore").astype("string")
    cleaned = cleaned.str.replace(r"[^A-ZN ]", "", regex=True)
    return cleaned.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_sex(series):
    cleaned = basic_text_clean(series)
    return cleaned.replace({"HOMBRE": "M", "MUJER": "F", "MASCULINO": "M",
                            "FEMENINO": "F", "": pd.NA})


for df in (fonasa, suseso):
    for col in ["nombre", "ap1", "ap2"]:
        df[f"{col}_clean"] = standardise_name(df[col])
    df["sexo_clean"] = clean_sex(df["sexo"])
    df["nac_clean"] = basic_text_clean(df["nacionalidad"])

CLEAN = ["nombre_clean", "ap1_clean", "ap2_clean", "sexo_clean", "nac_clean"]
TRUE_MATCHES = 4500

print(f"fonasa: {len(fonasa):,} records | suseso: {len(suseso):,} records")
print(f"true matches present in the data: {TRUE_MATCHES:,}")

fonasa: 30,000 records | suseso: 27,000 records
true matches present in the data: 4,500


## 1. The scale problem

Comparing two files of sizes $n_1$ and $n_2$ means $n_1 \times n_2$ comparisons.
The number grows with the product, while the number of true matches grows only
with the smaller file. Larger data therefore does not just cost more — it makes
the problem *proportionally* harder, because the ratio of true matches to
comparisons collapses.

In [2]:
scenarios = [
    ("This toolkit's sample", len(fonasa), len(suseso)),
    ("Two municipal registers", 200_000, 150_000),
    ("A national register pair", 5_000_000, 3_000_000),
    ("Census against an administrative source", 18_000_000, 12_000_000),
]

rows = []
for label, n1, n2 in scenarios:
    pairs = n1 * n2
    # a generous 100,000 comparisons per second
    rows.append({
        "scenario": label,
        "left": f"{n1:,}",
        "right": f"{n2:,}",
        "comparisons": f"{pairs:,.0f}",
        "hours at 100k/sec": round(pairs / 100_000 / 3600, 1),
    })

pd.DataFrame(rows)

,scenario,left,right,comparisons,hours at 100k/sec
0,This toolkit's sample,"30,000","27,000","810,000,000",2.2
1,Two municipal registers,"200,000","150,000","30,000,000,000",83.3
2,A national register pair,"5,000,000","3,000,000","15,000,000,000,000",41666.7
3,Census against an administrative source,"18,000,000","12,000,000","216,000,000,000,000",600000.0


The last row is the everyday situation of a statistical office, and it is
completely out of reach. Blocking is not an optimisation; it is what makes the
task possible.

## 2. Measuring a blocking rule

A blocking rule is judged on three things, and you cannot have all of them.

**Reduction ratio (RR)** — how much of the work it removes.

$$RR = 1 - \frac{\text{candidate pairs}}{\text{all possible pairs}}$$

**Pair completeness (PC)** — how many of the true matches survive it. This is
the one that matters most, because it is a hard ceiling on recall.

$$PC = \frac{\text{true matches among candidates}}{\text{all true matches}}$$

**Pairs quality (PQ)** — how concentrated the true matches are among the
candidates.

$$PQ = \frac{\text{true matches among candidates}}{\text{candidate pairs}}$$

RR and PC pull against each other: a rule that removes more work removes more
matches. PC is normally the binding constraint, because losing a match here is
irreversible.

The function below computes all three **without ever building the pair list**,
by counting block sizes. That matters: on real data you cannot materialise the
pairs you are trying to avoid creating.

In [3]:
ALL_PAIRS = len(fonasa) * len(suseso)

# the 4,500 true pairs, as (left id, right id)
truth = fonasa[["unique_id", "true_person_id"]].merge(
    suseso[["unique_id", "true_person_id"]], on="true_person_id", suffixes=("_f", "_s")
)
TRUE_PAIRS = set(zip(truth["unique_id_f"], truth["unique_id_s"]))
assert len(TRUE_PAIRS) == TRUE_MATCHES


def block_key(df, fields):
    """Build a blocking key; records missing any component are not eligible."""
    key = df[fields[0]].astype("string")
    for f in fields[1:]:
        key = key + "|" + df[f].astype("string")
    return key


def evaluate_blocking(fields, label):
    """Reduction ratio, pair completeness and pairs quality, without building pairs."""
    kf = block_key(fonasa, fields)
    ks = block_key(suseso, fields)

    # candidate pairs = sum over shared key values of n_left * n_right
    cf = kf.dropna().value_counts()
    cs = ks.dropna().value_counts()
    shared = cf.index.intersection(cs.index)
    n_candidates = int((cf.loc[shared].astype("int64") * cs.loc[shared].astype("int64")).sum())

    # true matches retained = true pairs whose two records share a key
    key_f = {i: (v if pd.notna(v) else None) for i, v in zip(fonasa["unique_id"], kf)}
    key_s = {i: (v if pd.notna(v) else None) for i, v in zip(suseso["unique_id"], ks)}
    retained = sum(
        1 for a, b in TRUE_PAIRS
        if key_f[a] is not None and key_f[a] == key_s[b]
    )

    return {
        "rule": label,
        "candidate_pairs": n_candidates,
        "RR": round(1 - n_candidates / ALL_PAIRS, 4),
        "PC": round(retained / TRUE_MATCHES, 4),
        "PQ": round(retained / n_candidates, 6) if n_candidates else 0.0,
        "true_matches_retained": retained,
    }


print(f"all possible pairs: {ALL_PAIRS:,}")
print(f"true matches      : {TRUE_MATCHES:,}")

all possible pairs: 810,000,000
true matches      : 4,500


## 3. Simple blocking: one field

The most basic rule: two records are compared only if they agree on sex.

In [4]:
results = [evaluate_blocking(["sexo_clean"], "sex")]
pd.DataFrame(results)

,rule,candidate_pairs,RR,PC,PQ,true_matches_retained
0,sex,383064445,0.5271,0.8678,0.00001,3905


Almost all the true matches survive, and the workload has halved — which sounds
useful until you look at the absolute number. Halving 810 million leaves 400
million. A rule on a two-valued field cannot do better than that, because the
reduction ratio of a blocking key is governed by how many distinct values it has
and how evenly the records spread across them.

**A blocking key must be high-cardinality.** This is the same property that made
a field valuable in the Fellegi-Sunter table, showing up again in a different
role.

## 4. Conjunctive blocking: several fields at once

Requiring agreement on *several* fields (an AND) multiplies the number of blocks
and shrinks each one dramatically.

In [5]:
results.append(evaluate_blocking(["nombre_clean", "ap1_clean"], "given name AND first surname"))
results.append(evaluate_blocking(["ap1_clean", "ap2_clean"], "first surname AND second surname"))
pd.DataFrame(results)

,rule,candidate_pairs,RR,PC,PQ,true_matches_retained
0,sex,383064445,0.5271,0.8678,0.000010,3905
1,given name AND first surname,1291,1.0000,0.2540,0.885360,1143
2,first surname AND second surname,3546,1.0000,0.2764,0.350818,1244


The reduction ratio is now essentially 1: these rules discard well over 99.99% of
the work.

But look at pair completeness. Each rule keeps only a fraction of the true
matches, because it demands exact agreement on two error-prone fields — and we
already know from [chapter 2.4](nb04-fellegi-sunter-by-hand.ipynb) that genuine
matches agree exactly on a given name only about a third of the time.

A conjunctive rule inherits the weakness of *every* field in it. That is the
central danger of blocking: this rule looks superb by RR and would quietly
discard most of what you were looking for.

## 5. Relaxing one component

A common fix: keep the conjunction but weaken one part of it, so that the rule
tolerates error in the place error is most likely.

In [6]:
for df in (fonasa, suseso):
    df["nombre_initial"] = df["nombre_clean"].str[:1]

results.append(evaluate_blocking(
    ["nombre_initial", "ap1_clean", "ap2_clean"],
    "given-name initial AND both surnames"))
pd.DataFrame(results)

,rule,candidate_pairs,RR,PC,PQ,true_matches_retained
0,sex,383064445,0.5271,0.8678,0.000010,3905
1,given name AND first surname,1291,1.0000,0.2540,0.885360,1143
2,first surname AND second surname,3546,1.0000,0.2764,0.350818,1244
3,given-name initial AND both surnames,1338,1.0000,0.2607,0.876682,1173


Better completeness at almost the same reduction ratio. Blocking on an initial
rather than a whole name survives abbreviation, compound given names recorded
differently, and most typographical errors after the first character.

## 6. Phonetic blocking

Names that sound alike are often spelled differently — `GONZALES` and
`GONZALEZ`, `MUNOZ` and `MUNIOZ`. A phonetic encoding maps them to the same code,
so blocking on the code catches pairs that blocking on the raw value misses.

Double Metaphone is the usual choice for Latin-script names.

In [7]:
import phonetics


def dmeta(value):
    if pd.isna(value) or value == "":
        return pd.NA
    try:
        return phonetics.dmetaphone(str(value))[0]
    except Exception:
        return pd.NA


for df in (fonasa, suseso):
    df["ap1_dm"] = df["ap1_clean"].map(dmeta)
    df["ap2_dm"] = df["ap2_clean"].map(dmeta)

print("examples of the encoding:")
for name in ["GONZALEZ", "GONZALES", "MUNOZ", "MUNIOZ", "SMITH", "SMYTH"]:
    print(f"  {name:<10} -> {dmeta(name)}")

examples of the encoding:
  GONZALEZ   -> KNSLS
  GONZALES   -> KNSLS
  MUNOZ      -> MNS
  MUNIOZ     -> MNS
  SMITH      -> SM0
  SMYTH      -> SM0


In [8]:
results.append(evaluate_blocking(["ap1_dm", "ap2_dm"], "both surnames, phonetic"))
pd.DataFrame(results)

,rule,candidate_pairs,RR,PC,PQ,true_matches_retained
0,sex,383064445,0.5271,0.8678,0.000010,3905
1,given name AND first surname,1291,1.0000,0.2540,0.885360,1143
2,first surname AND second surname,3546,1.0000,0.2764,0.350818,1244
3,given-name initial AND both surnames,1338,1.0000,0.2607,0.876682,1173
4,"both surnames, phonetic",7998,1.0000,0.3836,0.215804,1726


Compare this with the exact-agreement version of the same rule in section 4:
phonetic encoding recovers additional true matches, at the cost of larger blocks
and therefore more candidate pairs.

That is the trade in its usual form. You are not choosing between right and
wrong, you are choosing where to spend compute in exchange for recall.

## 7. Disjunctive blocking: the union of several rules

No single rule is good enough. A record whose given name is mistyped is lost by
any rule using the given name; a record whose second surname is missing is lost
by any rule using it.

The answer is to run **several rules and take the union** (an OR). A pair
survives if *any* rule keeps it, so a pair is only lost if it fails every rule at
once — which is far less likely than failing one.

In [9]:
RULE_SETS = [
    ("nombre_clean", "ap1_clean"),
    ("ap1_clean", "ap2_clean"),
    ("nombre_initial", "ap1_clean", "ap2_clean"),
    ("ap1_dm", "ap2_dm"),
]


def candidate_pairs(fields):
    """Materialise the candidate pairs for one rule (safe at this data size)."""
    l = fonasa[["unique_id"] + list(fields)].dropna()
    r = suseso[["unique_id"] + list(fields)].dropna()
    merged = l.merge(r, on=list(fields), suffixes=("_f", "_s"))
    return set(zip(merged["unique_id_f"], merged["unique_id_s"]))


union = set()
for fields in RULE_SETS:
    union |= candidate_pairs(fields)

retained = len(union & TRUE_PAIRS)
results.append({
    "rule": f"UNION of the {len(RULE_SETS)} rules above",
    "candidate_pairs": len(union),
    "RR": round(1 - len(union) / ALL_PAIRS, 4),
    "PC": round(retained / TRUE_MATCHES, 4),
    "PQ": round(retained / len(union), 6),
    "true_matches_retained": retained,
})
pd.DataFrame(results)

,rule,candidate_pairs,RR,PC,PQ,true_matches_retained
0,sex,383064445,0.5271,0.8678,0.000010,3905
1,given name AND first surname,1291,1.0000,0.2540,0.885360,1143
2,first surname AND second surname,3546,1.0000,0.2764,0.350818,1244
3,given-name initial AND both surnames,1338,1.0000,0.2607,0.876682,1173
4,"both surnames, phonetic",7998,1.0000,0.3836,0.215804,1726
5,UNION of the 4 rules above,8207,1.0000,0.3944,0.216279,1775


The union does what it was supposed to: higher pair completeness than any of its
components, while the reduction ratio stays effectively at 1.

This is why real linkage pipelines specify several blocking rules rather than
one. Splink takes a list of them and unions the results automatically.

## 8. The recall ceiling

Now the most important number in this notebook.

Pair completeness is not a diagnostic. It is a **hard ceiling on the recall of
everything downstream**. A true match whose records never end up in the same
block is never scored, never seen by any model, and cannot be recovered by
lowering a threshold.

In [10]:
chosen = results[-1]          # the union of the four rules

print(f"Chosen rule set  : {chosen['rule']}")
print(f"Candidate pairs  : {chosen['candidate_pairs']:,}")
print(f"Pair completeness: {chosen['PC']:.4f}")
print()
print(f"Of {TRUE_MATCHES:,} true matches:")
print(f"  {chosen['true_matches_retained']:,} can still be found by a model")
print(f"  {TRUE_MATCHES - chosen['true_matches_retained']:,} are already lost, "
      f"whatever the model does")
print()
print(f"Maximum achievable recall from here: {chosen['PC']:.1%}")

Chosen rule set  : UNION of the 4 rules above
Candidate pairs  : 8,207
Pair completeness: 0.3944

Of 4,500 true matches:
  1,775 can still be found by a model
  2,725 are already lost, whatever the model does

Maximum achievable recall from here: 39.4%


Note what we did *not* do: choose the rule with the highest pair completeness.
That would have been blocking on sex, at PC 0.87 — and 383 million candidate
pairs, which is no reduction worth having.

**Pair completeness on its own does not choose a rule. It chooses a rule subject
to a candidate count you can afford.** The union is the best completeness
available at a feasible cost.

Write that number down and carry it forward. When a model in
chapter 2.8 reports a recall, it has to be read against this
ceiling and not against 1.0 — the shortfall below the ceiling is the model's
fault, and everything above it is blocking's.

Reporting recall without stating the blocking ceiling is one of the most common
ways a linkage quality statement misleads its reader.

A ceiling in the high thirties is low. If that is not good enough for the use
case — and for most uses it would not be — the answer is **not** a better model.
It is another blocking rule that catches the kind of record the current set
loses.

### Raising the ceiling

So let us actually do that rather than assert it. The current rules all demand
exact agreement on at least one full name field. Records whose given name *and*
one surname are both corrupted fail every one of them.

Two more rules, built to fail differently: phonetic codes on the given name and
first surname, and the given-name initial with a phonetic first surname.

In [11]:
for df in (fonasa, suseso):
    df["nombre_dm"] = df["nombre_clean"].map(dmeta)

EXTRA_RULES = [
    ("nombre_dm", "ap1_dm"),
    ("nombre_initial", "ap1_dm", "ap2_dm"),
]

wider = set(union)
for fields in EXTRA_RULES:
    wider |= candidate_pairs(fields)

retained_wider = len(wider & TRUE_PAIRS)

print(f"before: {len(union):,} candidate pairs, PC {len(union & TRUE_PAIRS)/TRUE_MATCHES:.4f}")
print(f"after : {len(wider):,} candidate pairs, PC {retained_wider/TRUE_MATCHES:.4f}")
print()
print(f"ceiling raised from {len(union & TRUE_PAIRS)/TRUE_MATCHES:.1%} "
      f"to {retained_wider/TRUE_MATCHES:.1%}, "
      f"at {len(wider)/len(union):.1f}x the candidate pairs")

before: 8,207 candidate pairs, PC 0.3944
after : 8,868 candidate pairs, PC 0.4182

ceiling raised from 39.4% to 41.8%, at 1.1x the candidate pairs


That is the procedure in miniature: identify the failure mode the current rules
share, add a rule that does not share it, and measure the ceiling again. The cost
is more candidate pairs, which is the currency blocking trades in.

Keep iterating until the ceiling is acceptable or the candidate count stops being
affordable. Those are the only two constraints.

## 9. A different idea: sorted neighbourhood

All the rules above are **equality** rules: same block or not. The sorted
neighbourhood method works differently. It sorts all records by a key and
compares each record with the *w* records nearest to it in the sort order.

Its appeal is that it degrades gracefully. A record whose key is slightly wrong
lands slightly out of position rather than in a different block, and its true
partner may still be inside the window.

In [12]:
import recordlinkage as rl

left = fonasa.set_index("unique_id")
right = suseso.set_index("unique_id")

indexer = rl.Index()
indexer.sortedneighbourhood("ap1_clean", window=5)
snm = indexer.index(left, right)

snm_pairs = set(snm)
retained_snm = len(snm_pairs & TRUE_PAIRS)

print(f"candidate pairs      : {len(snm_pairs):,}")
print(f"reduction ratio      : {1 - len(snm_pairs)/ALL_PAIRS:.4f}")
print(f"pair completeness    : {retained_snm/TRUE_MATCHES:.4f}")
print(f"true matches retained: {retained_snm:,}")

candidate pairs      : 1,299,696
reduction ratio      : 0.9984
pair completeness    : 0.4562
true matches retained: 2,053


This is a genuinely different point on the trade-off, and not the one that might
have been expected.

Sorting on the first surname with a window of five reaches a *higher* pair
completeness than the union of equality rules — but it gets there by generating
around 1.3 million candidate pairs instead of eight thousand, roughly 160 times
as many. It buys recall with compute, at a poor exchange rate on this data.

Whether that trade is worth taking depends on your budget, and the two approaches
are not mutually exclusive: sorted neighbourhood can be added to a union of
equality rules as one more rule. Its characteristic strength is keys with a
meaningful *ordering* — dates, numeric identifiers, concatenated multi-field keys
— where being slightly wrong means being slightly out of position rather than in
the wrong block entirely.

The union of well-chosen equality rules remains the standard starting point, and
it is the approach Splink is built around.

## 10. How to choose

A workable procedure:

1. **Start from pair completeness, not reduction ratio.** Decide the recall
   ceiling your use case can live with. Everything else is negotiable; this is
   not.
2. **Use several rules and union them.** One rule will always have a systematic
   blind spot. Rules that fail for *different* reasons cover each other.
3. **Include at least one rule that does not use your weakest field.** If given
   names are unreliable, at least one rule should not need one.
4. **Count candidate pairs before running anything.** A rule that generates a
   billion pairs is not a rule; find out before you wait for it.
5. **Never block on a field you would not stake the linkage on.** Blocking is an
   irreversible filter. It is stricter than any comparison you make later.

The rule set built in section 7 is carried forward into the Splink chapters.

Chapter 2.6 puts everything from 2.2 to 2.5 together
in a real library: blocking rules, comparisons that recognise partial agreement,
and parameter estimation that does not need a ground truth.